## Data reading

In [0]:
from pyspark.sql.functions import *
df_cus = spark.read.table('databricks_cata.gold.dimcustomers').select('customer_id','DimCustomerKey').withColumnRenamed('customer_id', 'dim_customer_id')
df_pros = spark.read.table('databricks_cata.gold.dimproducts').select(col('product_id').alias('DimProductKey'), col('product_id')).withColumnRenamed('product_id', 'dim_product_id')
df = spark.read.table('databricks_cata.silver.orders_silver')

df_final = df.join(df_cus, df.customer_id == df_cus.dim_customer_id, 'left').join(df_pros, df.product_id == df_pros.dim_product_id, 'left')
df_final = df_final.drop('dim_customer_id','dim_product_id')
display(df_final)

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists('databricks_cata.gold.FactOrders'):
    target = DeltaTable.forPath(spark, 'abfss://gold@databrickse2esa.dfs.core.windows.net/FactOrders')
    target.alias('trg').merge(df_final.alias('src'), "trg.order_id = src.order_id")\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
                .execute()
else:
        df_final.write.mode('overwrite').format('delta').option('path','abfss://gold@databrickse2esa.dfs.core.windows.net/FactOrders')\
        .saveAsTable("databricks_cata.gold.FactOrders")

display(spark.read.table('databricks_cata.gold.FactOrders'))